In [1]:
import wandb
api = wandb.Api()
import numpy as np

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [3]:
runs = api.runs("ramithx1/Batch-Run-Maximum-Parsimony-A100-META-optimized")

In [4]:
errors = {}

seed = {}

sankoffs = {}
elves = {}

for l in [4,8,16,32,64,128]:
    errors[str(l)] = []
    sankoffs[str(l)] = []
    elves[str(l)] = []
    seed[str(l)] = []
    
    for run in runs:
        if(run.config['n_leaves'] == l and run.config['tLs']==[0, 0.005, 10, 50]):
            
            if("custom" in run.name or "XAblation" in run.name or 'best cost' not in run.summaryMetrics.keys()):
                continue
                
            if("Ablation" not in run.name):
                continue
            
            # print(run.config['epochs'])
            
            cost = run.summaryMetrics['best cost'] #run.summaryMetrics['traversal cost']
            gt   = run.summaryMetrics['sankoff_cost'] 


            errors[str(l)].append(cost - gt)
            sankoffs[str(l)].append(gt)
            elves[str(l)].append(cost)
            seed[str(l)].append(run.config['seed'])
            

            # print(run.summaryMetrics.keys())
            # print(run.summaryMetrics['sankoff_cost'])
            # print(run.summaryMetrics['traversal cost'])
            # print(run.history)
        


In [5]:
seed

{'4': [51, 50, 49, 48, 47, 46, 45, 44, 43, 42],
 '8': [46, 45, 44, 43, 42],
 '16': [46, 45, 44, 43, 42],
 '32': [47, 46, 45, 44, 43],
 '64': [44, 46, 45, 43, 42],
 '128': [46, 45, 44, 43, 42]}

In [6]:
sankoffs

{'4': [273, 278, 282, 277, 278, 276, 285, 271, 278, 274],
 '8': [646, 668, 645, 662, 648],
 '16': [1402, 1421, 1401, 1420, 1396],
 '32': [2916, 2918, 2916, 2911, 2930],
 '64': [5896, 5926, 5929, 5929, 5938],
 '128': [11960, 11968, 11979, 11941, 11963]}

In [7]:
errors

{'4': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 '8': [0, 0, 0, 0, 0],
 '16': [0, 12, 0, 0, 0],
 '32': [164, 119, 187, 104, 150],
 '64': [939, 951, 925, 1041, 951],
 '128': [10979, 2201, 8748, 12188, 2535]}

In [8]:
elves

{'4': [273, 278, 282, 277, 278, 276, 285, 271, 278, 274],
 '8': [646, 668, 645, 662, 648],
 '16': [1402, 1433, 1401, 1420, 1396],
 '32': [3080, 3037, 3103, 3015, 3080],
 '64': [6835, 6877, 6854, 6970, 6889],
 '128': [22939, 14169, 20727, 24129, 14498]}

In [9]:
for key in [4,8,16,32,64,128]:

    err = np.array(errors[str(key)])
    s_gt = np.array(sankoffs[str(key)])
    sol  = np.array(elves[str(key)])
    
    print(key, end = " -> ")
    assert err.shape == sol.shape
    assert s_gt.shape == sol.shape
    print(f" average sankoff = {s_gt.mean()}, avg ans = {sol.mean()}, avg error = {err.mean()}, std = {err.std()}, % error = {err.mean()/s_gt.mean()*100}") 

    

4 ->  average sankoff = 277.2, avg ans = 277.2, avg error = 0.0, std = 0.0, % error = 0.0
8 ->  average sankoff = 653.8, avg ans = 653.8, avg error = 0.0, std = 0.0, % error = 0.0
16 ->  average sankoff = 1408.0, avg ans = 1410.4, avg error = 2.4, std = 4.800000000000001, % error = 0.17045454545454544
32 ->  average sankoff = 2918.2, avg ans = 3063.0, avg error = 144.8, std = 30.022658110167395, % error = 4.961962853813995
64 ->  average sankoff = 5923.6, avg ans = 6885.0, avg error = 961.4, std = 40.937024806402334, % error = 16.229995273144706
128 ->  average sankoff = 11962.2, avg ans = 19292.4, avg error = 7330.2, std = 4200.5893586495695, % error = 61.27802578121081
